# G1Nav on Colab (free T4)

Runs the whole pipeline. **Runtime → Change runtime type → T4 GPU** first.

Every stage writes to Google Drive and skips work that is already done, so after a
disconnect: reconnect, re-run the **Setup** cells (1–3), then continue from the stage you were on.

| Stage | What | Rough T4 time (estimate, measure your own) |
|---|---|---|
| 4 | Train walker (PPO, 200M steps) | 2–5 h, resumable |
| 5 | Sim-to-sim walker check | 5 min |
| 6 | Expert data (600 episodes) | 20–40 min |
| 7 | GR00T features | 15–25 min |
| 8 | Behaviour cloning | 15–30 min |
| 9 | DAgger ×2 | 2 × ~45 min |
| 10 | Evaluation + videos | 30–60 min |

## 1. Drive + code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = 'https://github.com/alinaakh/LD_task.git'
try:  # private repo: add a GitHub token as the Colab secret GITHUB_TOKEN (key icon, left sidebar)
    from google.colab import userdata
    REPO_URL = REPO_URL.replace('https://', f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass
DATA = '/content/drive/MyDrive/g1nav_runs'                 # everything persistent lives here
os.makedirs(DATA, exist_ok=True)
os.environ['G1NAV_DATA'] = DATA
os.environ['MUJOCO_GL'] = 'egl'

if not os.path.exists('/content/g1nav'):
    !git clone -q {REPO_URL} /content/g1nav
CODE = '/content/g1nav/code'
%cd {CODE}
!git -C /content/g1nav log --oneline -1  # code version used for this run

## 2. Install
System Python keeps Colab's PyTorch; the walker's JAX stack goes into its own virtualenv.

In [ ]:
!pip -q install -r requirements.txt
!pip -q install uv
!test -x /content/jaxenv/bin/python || (uv venv -q /content/jaxenv && uv pip install -q --python /content/jaxenv/bin/python -r requirements-walker.txt)
!/content/jaxenv/bin/python -c "import jax; print('jax', jax.__version__, jax.devices())"
!python -c "import torch, mujoco; print('torch', torch.__version__, torch.cuda.get_device_name(0), '| mujoco', mujoco.__version__)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Robot assets + expert sanity test

In [ ]:
!python scripts/setup_assets.py
!python tests/test_expert_kinematic.py

## 4. Train the walker (PPO in MuJoCo Playground)
Resumable: if Colab disconnects, re-run setup and this cell; it continues from the last checkpoint.
Watch `reward` rise and `termination` go to ~0. `--num_timesteps` is the total across restarts.

In [ ]:
!/content/jaxenv/bin/python scripts/train_walker.py --out $G1NAV_DATA/walker --num_timesteps 200000000 2>&1 | grep -v -E "^(WARNING|I0000|W0000)"

## 5. Sim-to-sim check (plain MuJoCo, our arena)
Expect 0 falls and high expert success. If not, train the walker longer before generating data.

In [ ]:
!python scripts/check_walker.py --walker $G1NAV_DATA/walker/walker.npz --trials 10 --tasks 20 --video /content/walker_check.mp4
from IPython.display import Video
Video('/content/walker_check.mp4', embed=True, width=800)

## 6. Expert demonstrations

In [ ]:
!python scripts/generate_data.py --walker $G1NAV_DATA/walker/walker.npz --out $G1NAV_DATA/episodes/expert --n 600 --base_seed 0

## 7. Frozen GR00T features
The reused GR00T tensors (~3.1 GB) go to local disk, not Drive; they are re-downloaded per session.

In [ ]:
SUBSET = '/content/groot/groot_n16_subset.safetensors'
!python scripts/extract_features.py --dirs $G1NAV_DATA/episodes/expert --subset {SUBSET}

## 8. Behaviour cloning

In [ ]:
!python scripts/train_vla.py --data $G1NAV_DATA/episodes/expert --out $G1NAV_DATA/vla/bc --steps 30000

## 9. DAgger (two rounds)
The student drives, the expert labels; then retrain on everything.

In [ ]:
!python scripts/dagger.py --student $G1NAV_DATA/vla/bc/student.pt --walker $G1NAV_DATA/walker/walker.npz --out $G1NAV_DATA/episodes/dagger1 --n 200 --base_seed 1000000 --subset {SUBSET}
!python scripts/train_vla.py --data $G1NAV_DATA/episodes/expert $G1NAV_DATA/episodes/dagger1 --init $G1NAV_DATA/vla/bc/student.pt --out $G1NAV_DATA/vla/dagger1 --steps 15000

In [ ]:
!python scripts/dagger.py --student $G1NAV_DATA/vla/dagger1/student.pt --walker $G1NAV_DATA/walker/walker.npz --out $G1NAV_DATA/episodes/dagger2 --n 200 --base_seed 2000000 --subset {SUBSET}
!python scripts/train_vla.py --data $G1NAV_DATA/episodes/expert $G1NAV_DATA/episodes/dagger1 $G1NAV_DATA/episodes/dagger2 --init $G1NAV_DATA/vla/dagger1/student.pt --out $G1NAV_DATA/vla/dagger2 --steps 15000

## 10. Evaluation + videos
Run the expert too: it is the upper bound for the report.

In [ ]:
FINAL = f"{DATA}/vla/dagger2/student.pt"
!python scripts/evaluate.py --student {FINAL} --walker $G1NAV_DATA/walker/walker.npz --subset {SUBSET} --out $G1NAV_DATA/eval --n 20 --videos 3
!python scripts/evaluate.py --expert --walker $G1NAV_DATA/walker/walker.npz --out $G1NAV_DATA/eval_expert --n 20 --videos 0

In [ ]:
import glob
from IPython.display import Video, display
for v in sorted(glob.glob(f"{DATA}/eval/videos/*.mp4"))[:4]:
    print(v); display(Video(v, embed=True, width=900))

## 11. Package the submission
Upload your `report.pdf` to Drive first.

In [ ]:
!python scripts/package_submission.py --name YOUR_NAME --report /content/drive/MyDrive/report.pdf --student {FINAL} --eval $G1NAV_DATA/eval --out_dir /content/drive/MyDrive